In [ ]:
from pathlib import Path

import deepecohab as deh

In [ ]:
data_dir = Path("data")

files = list(data_dir.iterdir())

In [ ]:
project = deh.Project.create(
	project_name="test_project",
	experimenter="Tester",
	location=Path("."),
	description="testing the new backend",
)
report = project.add_recordings(files)
names = list(project.data_catalog)

# Custom analysis params

Tune the pipeline's knobs instead of taking the defaults, and pick which recordings to run it on.

In [ ]:
params = deh.AnalysisParams(
	minimum_time=5.0,  # ignore meetings shorter than this, in seconds
	minimum_time_alone=15.0,
	extrapolation_limit=3600.0,  # cap position carry-forward at 1h instead of 12h
)

# names=None (the default) runs every recording in the project; pass a subset to
# only analyse the recordings you're interested in
subset = names[:1]

project.run_analysis(params=params, names=subset)

# Running only specific steps

`targets` runs just those steps plus whatever they depend on - everything else is left alone.
`DataFrameRegistry.list_available()` lists every registered step name.

In [ ]:
from deepecohab.core.data_model import DataFrameRegistry

print(DataFrameRegistry.list_available())

# the rest of the recordings haven't been analysed yet, so this builds only
# chasings_df and the dependencies it needs (match_df, main_df) for them -
# activity_df, ranking, pairwise_meetings etc. are left unbuilt
remaining = names[1:]
project.run_analysis(params=params, names=remaining, targets=["chasings_df"])

# Overwriting existing results

By default, `run_analysis` skips any step whose parquet already exists on disk. `subset` is
already fully built, so re-running it needs `overwrite=True` to actually redo the work -
e.g. after changing `params`. `remaining` still has un-built steps left over from the
`targets`-restricted run above, so those fill in regardless.

In [ ]:
project.run_analysis(params=params, overwrite=True)
project.generate_project_table()

# Plotting

`deh.plot` builds a figure from an analysed recording as the plain dict Plotly draws.
`deh.to_figure` turns it into a Plotly figure to show, restyle or save.
`deh.PlotRegistry.list_available()` lists every plot name.

In [ ]:
recording = project[names[0]]

deh.to_figure(deh.plot("activity-line", recording)).show()

Options are keyword arguments. A `PlotContext` keeps the tables it has read in memory, so
several plots of one recording don't load them again.

In [ ]:
context = deh.PlotContext.from_recording(recording)

figure = deh.plot("activity-bar", context, metric="visits", agg="mean", phase_type=["dark_phase"])
deh.to_figure(figure).show()

In [ ]:
figure = deh.to_figure(deh.plot("ranking-line", context, mode="stability"))
figure.update_layout(template="publication")  # or "light", "dark"
figure.write_html("ranking.html")  # interactive, opens in any browser
figure.show()

In [ ]:
project.close()